# Task 1 — Friend Recommendation (Link Prediction)

**Dataset.** A social network of 7,624 LastFM users from Asian countries, with 27,806
mutual-follower edges ([Last.FM Asia](https://snap.stanford.edu/data/feather-lastfm-social.html)).

**Your job.** Given a user `src` and a list of candidate users, rank the candidates so
that the user's real friend comes out on top.

**Setup.** The 27,806 edges were split 70/15/15 into train/val/test. For every positive
edge we picked one endpoint as the query source and sampled negative candidates from
users that `src` is *not* connected to anywhere in the full graph:

| split | queries | positives : negatives |
|-------|---------|-----------------------|
| train | 19,464  | 1 : 5                 |
| val   | 4,171   | 1 : 20                |
| test  | 4,171   | 1 : 20                |

**Metrics.** Hit@1 and MRR.

**What you implement.** The three methods of `LinkPredictor` in section 3. The parts you write are marked `TODO`. Everything
else is provided, only modify when necessary.

**Note.** Run the cells in order. Please follow the instructions provided in the comments. You are encouraged to use coding agents.

## 1. Setup

On Google Colab this downloads the data. Running from a local checkout, it finds
the files already there and downloads nothing.

In [9]:
import json
import urllib.request
from collections import defaultdict
from pathlib import Path

import networkx as nx
import numpy as np
import pandas as pd

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.neural_network import MLPClassifier

GITHUB_REPO = "antman9914/CSE60745-Practice"
BRANCH = "main"

DATA_DIR = Path("data/task1")
REQUIRED = ["split_stats.json", "lp_graph_obs.csv",
            "lp_train.csv", "lp_val.csv", "lp_test.csv"]
RANDOM_SEED = 0

DATA_DIR.mkdir(parents=True, exist_ok=True)
for name in REQUIRED:
    if not (DATA_DIR / name).exists():
        url = f"https://raw.githubusercontent.com/{GITHUB_REPO}/{BRANCH}/data/task1/{name}"
        print(f"downloading {name} ...")
        urllib.request.urlretrieve(url, DATA_DIR / name)

missing = [n for n in REQUIRED if not (DATA_DIR / n).exists()]
assert not missing, f"could not obtain: {missing}"
assert tuple(map(int, nx.__version__.split(".")[:2])) >= (3, 0), \
    f"this notebook needs networkx >= 3.0, found {nx.__version__}"
print(f"data ready in {DATA_DIR}/ | networkx {nx.__version__}, pandas {pd.__version__}")

pd.set_option("display.width", 120)
np.set_printoptions(precision=4, suppress=True)

downloading split_stats.json ...
downloading lp_graph_obs.csv ...
downloading lp_train.csv ...
downloading lp_val.csv ...
downloading lp_test.csv ...
data ready in data/task1/ | networkx 3.6.1, pandas 2.2.3


## 2. Data loading

In [11]:
def load_task1_data(data_dir=DATA_DIR):
    """Load the two observable graphs and the three query sets.

    Returns
    -------
    graphs : dict[str, networkx.Graph]
        "train"    - all 7,624 nodes and the 70% of edges in the training split.
        "trainval" - the same, plus the 15% of edges in the validation split.
        Validation and test positive edges are absent from "train"; test positive edges
        are absent from "trainval". These are the only two graphs you may look at.
    splits : dict[str, pandas.DataFrame]
        One frame per split with columns query_id, src, dst, label. All rows sharing a
        query_id form one ranking problem with exactly one positive candidate.
    """
    stats = json.loads((data_dir / "split_stats.json").read_text())
    splits = {name: pd.read_csv(data_dir / f"lp_{name}.csv")
              for name in ("train", "val", "test")}

    train_edges = pd.read_csv(data_dir / "lp_graph_obs.csv")[["src", "dst"]]
    val_edges = splits["val"].loc[splits["val"]["label"] == 1, ["src", "dst"]]

    G_train = nx.Graph()
    G_train.add_nodes_from(range(stats["n_nodes"]))
    G_train.add_edges_from(train_edges.itertuples(index=False, name=None))

    G_trainval = G_train.copy()
    G_trainval.add_edges_from(val_edges.itertuples(index=False, name=None))

    return {"train": G_train, "trainval": G_trainval}, splits


graphs, splits = load_task1_data()
G_train, G_trainval = graphs["train"], graphs["trainval"]
train_df, val_df, test_df = splits["train"], splits["val"], splits["test"]

for name, G in graphs.items():
    n_isolated = sum(1 for _, d in G.degree() if d == 0)
    print(f"G_{name:<9} {G.number_of_edges():>6} edges, {n_isolated:>4} isolated nodes, "
          f"{nx.number_connected_components(G):>4} connected components")
for name, df in splits.items():
    n_q = int(df["label"].sum())
    print(f"{name:>5}: {n_q:>6} queries x {len(df) // n_q:>2} candidates = {len(df):>6} rows")

train_df.head(6)


G_train      19464 edges,  666 isolated nodes,  734 connected components
G_trainval   23635 edges,  312 isolated nodes,  352 connected components
train:  19464 queries x  6 candidates = 116784 rows
  val:   4171 queries x 21 candidates =  87591 rows
 test:   4171 queries x 21 candidates =  87591 rows


,query_id,src,dst,label
0,0,6477,5004,0
1,0,6477,5092,0
2,0,6477,6243,0
3,0,6477,1834,0
4,0,6477,2467,0
5,0,6477,3038,1


## 3. Your implementation

Fill in the three methods in LinkPredictor.

`adjacency_lookups(G)` returns adjacency matrix ADJ and degree matrix DEG for input graph:

```python
ADJ, DEG = adjacency_lookups(G)    # ADJ[u] is a set of neighbours, DEG[u] an int
```


In [12]:
_LOOKUP_CACHE = {}


def adjacency_lookups(G):
    """Return (ADJ, DEG) for a graph: neighbour sets and degrees, cached per graph."""
    if G not in _LOOKUP_CACHE:
        adj = {u: set(G.neighbors(u)) for u in G}
        _LOOKUP_CACHE[G] = (adj, {u: len(adj[u]) for u in G})
    return _LOOKUP_CACHE[G]


In [23]:
from networkx.linalg.graphmatrix import adjacency_matrix
class LinkPredictor:
    """Rank candidate friends for a source user using graph structural properties."""

    # ==================================================================================
    # TODO 1 of 3 - feature engineering.
    # ==================================================================================
    def build_features(self, G, pairs):
      """Turn node pairs into numbers."""
      """Parameters
        ----------
        G : networkx.Graph
            The graph you may look at when describing these pairs. It is not the same
            graph for every split.
        pairs : numpy.ndarray of shape (n_pairs, 2)
            Each row is a candidate pair (src, dst) to be described. This is a whole
            split at a time, not a batch: n_pairs is 116,784 for train and 87,591 for
            val and for test. The pairs arrive flat; the framework regroups your output
            by query before handing it to fit() and score().

        Returns
        -------
        numpy.ndarray of shape (n_pairs, n_features), dtype float
            One row of features per input pair. The number of columns is up to you.

        Notes
        -----
        This method is called three times, once per split, and not always with the same
        graph. Please use `adjacency_lookups(G)` to obtain the adjacency and degree
        lookups for whichever graph you were given.
        """

      #step 1: get neighbor list and degree count
      name_of_friends, num_of_friends = adjacency_lookups(G)

      #step2: list to put features of each (person pair)
      # feature list will be a row in matrix
      features_of_girls = []

      #step3: find each feature for each person pair
      for girly1, girly2 in pairs:
        #find their friends in order to see what friends they have in common
        friends_of_girly1 = name_of_friends[girly1]
        friends_of_girly2 = name_of_friends[girly2]

        #feature 1: num of mutual friends of person pair
        mutual_friends = len(friends_of_girly1 & friends_of_girly2)

        #feature 2: num of friends for each person in pair
        girly1_friends = num_of_friends[girly1]
        girly2_friends = num_of_friends[girly2]

        #feature3: check for few friends for each person in pair
        is_girly1_loner = 1 if num_of_friends[girly1] <= 2 else 0
        is_girly2_loner = 1 if num_of_friends[girly2] <= 2 else 0

        #put features in the feature list
        features_of_girls.append([
            mutual_friends,
            girly1_friends,
            girly2_friends,
            is_girly1_loner,
            is_girly2_loner
        ])
        #turn list into numpy array for matrix

      return np.array(features_of_girls, dtype=float)

    # ==================================================================================
    # TODO 2 of 3 - training.
    # ==================================================================================
    def fit(self, X_train, y_train, X_val, y_val):
        """Train a classifier or ranking model on the training pairs. """
        #step 1: reshape training data & validation data
        n_queries, n_candidates, n_features = X_train.shape

        X_train_flat = X_train.reshape(-1, n_features)
        y_train_flat = y_train.ravel()

        #step 2: pick a classifier. i picked logistic regression
        self.friend_prediction = LogisticRegression(max_iter=1000)

        #train model
        self.friend_prediction.fit(X_train_flat, y_train_flat)

        """The features are grouped by query, so axis 0 is a query, axis 1 is that query's
        candidates, and axis 2 is your features.

        Parameters
        ----------
        X_train : numpy.ndarray of shape (n_queries, n_candidates, n_features)
            The entire training set: 19,464 queries, 6 candidates each. n_features is
            however many columns your build_features returned.
            The positive candidate is ALWAYS at index 0 along axis 1.
        y_train : numpy.ndarray of shape (n_queries, n_candidates)
            1 for the real edge, 0 for a sampled non-edge. Because of the convention
            above this is always [1, 0, 0, 0, 0, 0] for every query; it is passed anyway
            so that flattening stays convenient.
        X_val : numpy.ndarray of shape (n_queries, n_candidates, n_features)
            The entire validation set: 4,171 queries, 21 candidates each.
            Here the positive is at an UNKNOWN position, exactly as it will be at test
            time. Use y_val to find it.
        y_val : numpy.ndarray of shape (n_queries, n_candidates)
            One 1 per row, at the position of the real edge.

        Notes
        -----
        Store the fitted estimator on self (for example self.model) so that score() can
        use it.
        """

    # ==================================================================================
    # TODO 3 of 3 - inference.
    # ==================================================================================
    def score(self, X):
      """Score candidate pairs."""

      num__queries, num_candidates, num_features = X.shape
      #flaten input
      X_train_flattened = X.reshape(-1, num_features)

      #what is the probability each person-pair are actually friends
      friendship_probablity = self.friend_prediction.predict_proba(X_train_flattened)[:, 1]
      return friendship_probablity.reshape(num__queries, num_candidates)

      """Called twice, once for validation and once for test, each time with that split's
        complete feature tensor. These are not batches.

        Parameters
        ----------
        X : numpy.ndarray of shape (n_queries, n_candidates, n_features)
            (4171, 21, n_features) for both calls. n_features is the same as in fit().
            The positive is at an unknown position along axis 1 — do not assume index 0
            here, that convention applies to the training split only.

        Returns
        -------
        numpy.ndarray of shape (n_queries, n_candidates)
            One score per candidate, in the same layout as X. A HIGHER score must mean
            "more likely to be a real edge"; the evaluation ranks the candidates of each
            query by this value in descending order.

        Notes
        -----
        Return a continuous score, not a hard 0/1 prediction.
        """

## 4. Evaluation Toolset

In [24]:
def evaluate_ranking(df, scores, seed=RANDOM_SEED):
    """Compute Hit@1 and MRR within each query.

    Candidates are shuffled before sorting so that ties are broken uniformly at random.
    This matters: a model that gives every candidate the same score should land on the
    random baseline, not on whatever order the file happened to store them in.
    """
    rng = np.random.default_rng(seed)
    d = df[["query_id", "label"]].copy()
    d["score"] = scores
    d["tiebreak"] = rng.random(len(d))

    d = d.sort_values(["query_id", "score", "tiebreak"], ascending=[True, False, True])
    d["rank"] = d.groupby("query_id").cumcount() + 1

    pos = d.loc[d["label"] == 1, ["query_id", "rank"]]
    metrics = {
        "n_queries": len(pos),
        "hit@1": float((pos["rank"] == 1).mean()),
        "MRR": float((1.0 / pos["rank"]).mean()),
    }
    return metrics, pos.set_index("query_id")["rank"]


def breakdown_by_source_degree(df, ranks, degrees, bins=(0, 1, 2, 4, 8, 16, np.inf)):
    """Split Hit@1 and MRR by how many friends the source user has in the graph.

    `degrees` is a node -> degree mapping for the graph that split was scored on.
    """
    pos = df[df["label"] == 1].set_index("query_id")
    d = pd.DataFrame({"rank": ranks})
    d["src_degree"] = pos.loc[d.index, "src"].map(degrees).to_numpy()
    d["bucket"] = pd.cut(d["src_degree"], bins=list(bins), right=False)

    return d.groupby("bucket", observed=True).agg(
        n_queries=("rank", "size"),
        hit_at_1=("rank", lambda r: (r == 1).mean()),
        MRR=("rank", lambda r: (1.0 / r).mean()),
    ).round(4)


def breakdown_both(scores):
    """Per-degree breakdown of validation and test, each on the graph it was scored on.

    Test queries are described on G_trainval, so they are bucketed by degree in that
    graph rather than in G_train: the validation edges rescue 354 of the 666 nodes that
    are isolated in G_train, and using the wrong map would put them in the wrong row.
    """
    _, val_ranks = evaluate_ranking(val_df, scores["val"])
    _, test_ranks = evaluate_ranking(test_df, scores["test"])
    return {
        "val": breakdown_by_source_degree(val_df, val_ranks, dict(G_train.degree())),
        "test": breakdown_by_source_degree(test_df, test_ranks, dict(G_trainval.degree())),
    }


def compare_breakdowns(before, after):
    """Hit@1 per degree bucket, before against after, for validation and test."""
    cols = {}
    for split in ("val", "test"):
        b, a = before[split], after[split]
        cols[(split, "n")] = b["n_queries"]
        cols[(split, "before")] = b["hit_at_1"]
        cols[(split, "after")] = a["hit_at_1"]
        cols[(split, "delta")] = (a["hit_at_1"] - b["hit_at_1"]).round(4)
    return pd.DataFrame(cols)


## 5. Running the pipeline

In [25]:
# Which graph each split is described on. Training and validation queries see the
# training edges only; test queries additionally see the validation edges.
FEATURE_GRAPH = {"train": "train", "val": "train", "test": "trainval"}


def to_query_tensor(df, X, positive_first=False):
    """Group a flat feature matrix by query: (n_pairs, F) -> (n_queries, n_candidates, F).

    The rows of a split are contiguous per query and every query in a split has the same
    number of candidates, so this is a reshape plus, optionally, a permutation inside
    each query.

    With `positive_first`, the single positive candidate is moved to index 0 along axis 1
    and the negatives keep their relative order. This is used for the training split
    only: at validation and test time the position of the positive must stay unknown,
    otherwise a model could "win" by always scoring index 0 highest.

    Returns the tensor, the matching label matrix, and the row indices used, so scores
    computed on the tensor can be mapped back to the row order of `df`.
    """
    labels = df["label"].to_numpy()
    n_cand = len(df) // int(labels.sum())
    n_q = len(df) // n_cand

    rows = np.arange(len(df)).reshape(n_q, n_cand)
    lab = labels.reshape(n_q, n_cand)
    if positive_first:
        # A stable sort on -label puts the one positive in column 0 and leaves the
        # negatives in the order they had in the file.
        cols = np.argsort(-lab, axis=1, kind="stable")
        rows = np.take_along_axis(rows, cols, axis=1)
        lab = np.take_along_axis(lab, cols, axis=1)
    return X[rows], lab, rows


def flatten_scores(scores, rows, n_pairs):
    """Map (n_queries, n_candidates) scores back to the row order of the split."""
    flat = np.empty(n_pairs, dtype=float)
    flat[rows.ravel()] = np.asarray(scores, dtype=float).ravel()
    return flat


def run_pipeline(model, graphs, splits):
    """Featurise each split on its own graph, group by query, fit on train, score."""
    tensors, labels, rows = {}, {}, {}
    for name in ("train", "val", "test"):
        df, G = splits[name], graphs[FEATURE_GRAPH[name]]
        X = np.asarray(model.build_features(G, df[["src", "dst"]].to_numpy()), dtype=float)
        assert X.shape[0] == len(df), (
            f"build_features returned {X.shape[0]} rows for {len(df)} pairs"
        )
        tensors[name], labels[name], rows[name] = to_query_tensor(
            df, X, positive_first=(name == "train")
        )
        print(f"{name:>5}: {tensors[name].shape}  "
              f"(graph: G_{FEATURE_GRAPH[name]}, {G.number_of_edges()} edges)")

    n_feat = {t.shape[2] for t in tensors.values()}
    assert len(n_feat) == 1, f"inconsistent feature count across splits: {n_feat}"

    model.fit(tensors["train"], labels["train"], tensors["val"], labels["val"])

    scores = {}
    for name in ("val", "test"):
        s = np.asarray(model.score(tensors[name]), dtype=float)
        assert s.shape == tensors[name].shape[:2], (
            f"score returned shape {s.shape}, expected {tensors[name].shape[:2]} "
            f"for the {name} split"
        )
        scores[name] = flatten_scores(s, rows[name], len(splits[name]))
    return scores


model = LinkPredictor()
scores = run_pipeline(model, graphs, splits)

train: (19464, 6, 5)  (graph: G_train, 19464 edges)
  val: (4171, 21, 5)  (graph: G_train, 19464 edges)
 test: (4171, 21, 5)  (graph: G_trainval, 23635 edges)


## 6. Final test evaluation

Run this **at the end**.

In [26]:
val_metrics, val_ranks = evaluate_ranking(val_df, scores["val"])
print("validation:", val_metrics)
test_metrics, test_ranks = evaluate_ranking(test_df, scores["test"])
print("validation:", val_metrics)
print("test:      ", test_metrics)

validation: {'n_queries': 4171, 'hit@1': 0.6020139055382402, 'MRR': 0.6839065074558304}
validation: {'n_queries': 4171, 'hit@1': 0.6020139055382402, 'MRR': 0.6839065074558304}
test:       {'n_queries': 4171, 'hit@1': 0.6619515703668185, 'MRR': 0.7349926638252321}


## 7. Cold Start Challenge: users with few friends

Your overall Hit@1 is an average over very different users. Ranking quality falls sharply as the
source user has fewer friends or even no friends at all.

In the code below, `breakdown_by_source_degree` splits the evaluation queries by the source user's degree, so you can check the phenomenon directly. `compare_breakdowns` puts two such tables side by
side, so after a change you can see which users it actually helped.

**TODO.** An inactive user has little graph structure around them, so pair features such
as the number of common neighbours come out nearly the same for every one of their candidates, leaving the ranking to the random tie-break.
Try to improve link prediction for these users: change your feature set and improve your model implemented in section 3,
re-run section 5 and 6, and come back here to compare. Judge the result per degree bucket rather
than on the overall number. Before modification, please run the first cell in this section to save the baseline results, and use the second cell to make comparison afterwards.


In [ ]:
baseline = breakdown_both(scores)

display(baseline["val"], baseline["test"])


In [ ]:
# After re-running sections 5 and 6, run this to see what your change did to each kind
# of user, on both the validation and the test split.
current = breakdown_both(scores)

compare_breakdowns(baseline, current)
